<a href="https://colab.research.google.com/github/MuskanAwais/emotion-recognition/blob/main/emotion_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision torchaudio
!pip install onnx onnxruntime
!pip install matplotlib tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 87.7 MB/s eta 0:00:00


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets download -d fahadullaha/facial-emotion-recognition-dataset

Dataset URL: https://www.kaggle.com/datasets/fahadullaha/facial-emotion-recognition-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 200M/200M [00:01<00:00, 121MB/s]



In [ ]:
!unzip facial-emotion-recognition-dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: processed_data/surprise/surprise_00920.jpg  
  inflating: processed_data/surprise/surprise_00921.jpg  
  inflating: processed_data/surprise/surprise_00922.jpg  
  inflating: processed_data/surprise/surprise_00923.jpg  
  inflating: processed_data/surprise/surprise_00924.jpg  
  inflating: processed_data/surprise/surprise_00925.jpg  
  inflating: processed_data/surprise/surprise_00926.jpg  
  inflating: processed_data/surprise/surprise_00927.jpg  
  inflating: processed_data/surprise/surprise_00928.jpg  
  inflating: processed_data/surprise/surprise_00929.jpg  
  inflating: processed_data/surprise/surprise_00930.jpg  
  inflating: processed_data/surprise/surprise_00931.jpg  
  inflating: processed_data/surprise/surprise_00932.jpg  
  inflating: processed_data/surprise/surprise_00933.jpg  
  inflating: processed_data/surprise/surprise_00934.jpg  
  inflating: processed_data/surprise/surprise_00935.jpg  
  inflating: processe

In [ ]:
import os

print(os.listdir("."))

['.config', 'processed_data', 'kaggle.json', 'facial-emotion-recognition-dataset.zip', 'sample_data']


In [ ]:
import os

base_path = "/content/processed_data"

for root, dirs, files in os.walk(base_path):
    print("ROOT:", root)
    print("DIRS:", dirs)
    print("FILES:", len(files))
    print("-" * 40)
    break

ROOT: /content/processed_data
DIRS: ['angry', 'happy', 'fear', 'disgust', 'surprise', 'neutral', 'sad']
FILES: 0
----------------------------------------


Create train test folder

In [ ]:
import os

base_dir = "/content/processed_data"

train_dir = "/content/dataset/train"
test_dir = "/content/dataset/test"

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

Split images (80% train, 20% test)

In [ ]:
import shutil
import random

classes = os.listdir(base_dir)

for cls in classes:
    class_path = os.path.join(base_dir, cls)

    images = os.listdir(class_path)
    random.shuffle(images)

    split = int(len(images) * 0.8)

    train_imgs = images[:split]
    test_imgs = images[split:]

    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(class_path, img),
                    os.path.join(train_dir, cls, img))

    for img in test_imgs:
        shutil.copy(os.path.join(class_path, img),
                    os.path.join(test_dir, cls, img))

print("Dataset split completed")

Dataset split completed


Verify new structure

In [ ]:
import os

print("Train folders:", os.listdir(train_dir))
print("Test folders:", os.listdir(test_dir))

Train folders: ['angry', 'happy', 'fear', 'disgust', 'surprise', 'neutral', 'sad']
Test folders: ['angry', 'happy', 'fear', 'disgust', 'surprise', 'neutral', 'sad']


Import libraries

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

STEP 9.2 — Define image preprocessing

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

STEP 9.3 — Load dataset

In [ ]:
train_path = "/content/dataset/train"
test_path = "/content/dataset/test"

train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
test_dataset = datasets.ImageFolder(test_path, transform=test_transform)

print("Classes:", train_dataset.classes)

Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


STEP 9.4 — Create DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Train samples: 39822
Test samples: 9957


STEP 10 — MobileNetV3 Model (TRANSFER LEARNING)

STEP 10.1 — Load pretrained model

In [ ]:
import torch.nn as nn
from torchvision import models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.mobilenet_v3_large(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 118MB/s] 


STEP 10.2 — Modify final layer

In [ ]:
num_classes = 7

model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

model = model.to(device)

STEP 10.3 — Loss + Optimizer

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

CHECK 1 — Visual batch test

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))

print("Batch shape:", images.shape)
print("Labels:", labels[:10])

Batch shape: torch.Size([32, 3, 224, 224])
Labels: tensor([4, 3, 4, 6, 5, 4, 6, 4, 2, 1])


STEP 11 — FULL TRAINING LOOP (real AI training starts + accuracy tracking + saving model)

⚙️ STEP 11.1 — Training setup

In [ ]:
import torch

epochs = 5  # start small (we can increase later)

best_acc = 0.0

STEP 11.2 — Training loop

In [ ]:
for epoch in range(epochs):
    model.train()

    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {total_loss/len(train_loader):.4f}")

Epoch [1/5] Loss: 1.0461
Epoch [2/5] Loss: 0.8526
Epoch [3/5] Loss: 0.7721
Epoch [4/5] Loss: 0.7159
Epoch [5/5] Loss: 0.6652


STEP 11.3 — Validation (accuracy check)

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print("Validation Accuracy:", acc)

Validation Accuracy: 73.20478055639249


STEP 11.4 — Save model

In [ ]:
torch.save(model.state_dict(), "emotion_model.pth")
print("Model saved!")

Model saved!


 STEP 12 — Export Model for Production (ONNX)

STEP 12.1 — Load trained model

In [ ]:
import torch
from torchvision import models
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.mobilenet_v3_large(pretrained=False)

model.classifier[3] = nn.Linear(model.classifier[3].in_features, 7)

model.load_state_dict(torch.load("emotion_model.pth", map_location=device))

model.eval()
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


STEP 12.2 — Create dummy input

In [ ]:
dummy_input = torch.randn(1, 3, 224, 224).to(device)

STEP 12.3 — Export to ONNX

In [ ]:
!pip install onnx onnxscript onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.5 MB/s eta 0:00:00


🚀 STEP 12.5 — QUICK MODEL TEST (VERY IMPORTANT)

In [ ]:
!pip install onnx onnxruntime

In [ ]:
torch.onnx.export(
    model,
    dummy_input,
    "emotion_model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11,
    export_params=True,
    do_constant_folding=True,
    training=torch.onnx.TrainingMode.EVAL
)

W0515 22:49:08.371000 10403 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0515 22:49:08.887000 10403 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0515 22:49:08.889000 10403 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ra

[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.10.0+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,3,224,224]>
            ),
            outputs=(
                %"output"<FLOAT,[1,7]>
            ),
            initializers=(
                %"features.0.0.weight"<FLOAT,[16,3,3,3]>{Tensor(...)},
                %"features.1.block.0.0.weight"<FLOAT,[16,1,3,3]>{Tensor(...)},
                %"features.1.block.1.0.weight"<FLOAT,[16,16,1,1]>{Tensor(...)},
                %"features.2.block.1.0.weight"<FLOAT,[64,1,3,3]>{Tensor(...)},
                %"features.3.block.1.0.weight"<FLOAT,[72,1,3,3]>{Tensor(...)},
                %"features.4.block.2.fc1.bias"<FLOAT,[24]>{TorchTensor(...)},
                %"features.4.block.2.fc2.bias"<FLOAT,[72]>{

In [ ]:
import os

# delete split file if created
if os.path.exists("emotion_model.onnx.data"):
    os.remove("emotion_model.onnx.data")

In [ ]:
import os
print(os.listdir())

['.config', 'emotion_model.onnx', 'emotion_model.pth', 'processed_data', 'kaggle.json', 'dataset', 'facial-emotion-recognition-dataset.zip', 'sample_data']


🔹 STEP 1 — Load a sample image

In [ ]:
import torch
from PIL import Image
from torchvision import transforms

classes = ['angry','disgust','fear','happy','neutral','sad','surprise']

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

🔹 STEP 2 — Load any test image

In [ ]:
img_path = "/content/dataset/test/happy"  # change folder if needed

import os

img_file = os.listdir(img_path)[0]
img_path = os.path.join(img_path, img_file)

image = Image.open(img_path).convert("RGB")
image = transform(image).unsqueeze(0).to(device)

STEP 3 — Predict

In [ ]:
model.eval()

with torch.no_grad():
    output = model(image)
    pred = torch.argmax(output, 1)

print("Predicted Emotion:", classes[pred.item()])

Predicted Emotion: happy


STEP 13 — FastAPI BACKEND (REAL AI API)

STEP 13.1 — Install FastAPI (Colab or Local)

In [ ]:
!pip install fastapi uvicorn python-multipart onnxruntime opencv-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print(os.listdir("."))

['.config', 'emotion_model.onnx', 'emotion_model.pth', 'emotion_model.onnx.data', 'processed_data', 'kaggle.json', 'dataset', 'facial-emotion-recognition-dataset.zip', 'sample_data']


In [ ]:
from google.colab import files

files.download("emotion_model.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

testing


In [ ]:
print(train_dataset.classes)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [ ]:
!apt-get install git

!git config --global user.name "MuskanAwais"
!git config --global user.email "2022cs38@student.uet.edu.pk"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [ ]:
!git clone https://github.com/MuskanAwais/emotion-recognition.git
%cd emotion-recognition

Cloning into 'emotion-recognition'...
/content/emotion-recognition/emotion-recognition


In [ ]:
!ls /content

emotion-recognition  sample_data
